# 03 — Gold Star Schema

**Layer:** Gold · **Day:** 1 · **Authoritative script:** `scripts/day1_build_lakehouse.py` (function `build_gold`)

## Objective

Turn the silver employee table into a **star schema** that the Power BI semantic model and the FastAPI Copilot can both consume. Gold is the contract presented to consumers — once they integrate against it, the upstream layers can change without breaking them.

## Inputs

- `lakehouse/silver/employees.parquet`

## Outputs (all under `lakehouse/gold/`)

### Dimensions

| File | Grain | Notes |
|---|---|---|
| `dim_employee.parquet` | one row per employee | demographic attributes only — no salary, no attrition |
| `dim_department.parquet` | one row per department | surrogate key `DepartmentID` |
| `dim_jobrole.parquet` | one row per (JobRole × JobLevel) | surrogate key `JobRoleID`, FK `DepartmentID` |
| `dim_date.parquet` | one row per month for 24 months | marked as the date table in Power BI |

### Facts

| File | Grain | Notes |
|---|---|---|
| `fact_employee_snapshot.parquet` | employee × month | synthetic monthly snapshots; `IsActive`, `AttritionThisMonth`, satisfaction, salary, tenure |
| `fact_recruitment.parquet` | application | synthetic funnel: Applied → Screened → Interviewed → Offered → Hired/Rejected |
| `fact_engagement_pulse.parquet` | employee × quarter | synthetic engagement pulses; `PulseScore`, `ThemesFlagged` |
| `fact_attrition_risk.parquet` | one row per employee | **written by Day 2** — the ML model writes its predictions back here |

## Business value

A star schema is the right shape for both Power BI (DirectQuery / DirectLake friendly) and ad-hoc SQL via DuckDB. Synthetic monthly snapshots are what make the time-series story possible — without them, a single static IBM dataset cannot answer "how did attrition trend by quarter?"

## Reproduce

```powershell
python scripts\day1_build_lakehouse.py
```

In [ ]:
from pathlib import Path
import pandas as pd

GOLD = Path('../lakehouse/gold')
files = sorted(GOLD.glob('*.parquet')) if GOLD.exists() else []
for f in files:
    print(f'{f.name:<35}  {len(pd.read_parquet(f)):>7,} rows')

In [ ]:
# Sanity check the snapshot grain — should be 1,470 employees × up to 24 months
if (GOLD / 'fact_employee_snapshot.parquet').exists():
    snap = pd.read_parquet(GOLD / 'fact_employee_snapshot.parquet')
    print('rows:', len(snap))
    print('unique employees:', snap['EmployeeID'].nunique())
    print('months:', snap['DateKey'].nunique())
    print('active rate by month (head):')
    print(snap.groupby('DateKey')['IsActive'].mean().round(3).head())

## Power BI relationships

The semantic model wires these tables together; the full list is in `powerbi/power_query_import_guide.md` §4. The headline relationships:

- `FactEmployeeSnapshot[EmployeeID]` → `DimEmployee[EmployeeID]`
- `FactEmployeeSnapshot[DateKey]` → `DimDate[DateKey]`
- `FactEmployeeSnapshot[DepartmentID]` → `DimDepartment[DepartmentID]`
- `FactEmployeeSnapshot[JobRoleID]` → `DimJobRole[JobRoleID]`
- `FactAttritionRisk[EmployeeID]` → `DimEmployee[EmployeeID]`

## Interview talking points

- The risk fact is written **into** Gold by the ML pipeline. That keeps the model output reusable and avoids tight coupling between the model and the dashboard.
- A star schema is what makes incremental moves to Fabric/OneLake painless — the relationships and DAX measures port over directly.
- Synthetic snapshots are flagged honestly in the model card and the README so claims about temporal trends stay calibrated.